# IGMC baseline (Zhang & Chen, 2020)

Implements Table 1's `IGMC (rand)` / `IGMC (supp)` rows: Inductive Matrix
Completion via Graph Neural Networks, scored under both naive abstention
rules from `abstention.py` (IGMC has no learned uncertainty of its own, same
as Plain MF in `experiments/mf_vs_ours.ipynb`).

**What IGMC does** (proposal ref [2]): instead of learning one global
embedding per user/item (like Plain MF), IGMC predicts a rating for `(u, i)`
purely from the *local enclosing subgraph* around that pair in the training
bipartite rating graph. This is what makes it inductive: a new user/item with
no learned embedding can still be scored as long as it has some rated
neighbours, because the model only ever sees graph structure, never a
per-node learned parameter.

For each target pair we build the 1-hop enclosing subgraph (every node
reachable within 1 hop of either the target user or the target item, every
training edge between two included nodes, with the target edge itself
excluded so the model can't just read off the label), one-hot-label each
node by its structural role, then run a small relational GCN (one weight
matrix per rating value, 1..5) over it. Concatenating every layer's
embedding of the two target nodes gives a fixed-size graph representation,
which an MLP maps to a predicted rating.

**Deliberate simplification vs. the paper**: the paper labels nodes with
*double-radius node labeling* (DRNL) - a pair of hop-distances to each
target, paired into one integer. Implementing DRNL faithfully only matters
once subgraphs go beyond 1 hop; at h=1 every non-target node's role is
already fully described by "which side is it on, and is it a target" - so we
use that 4-way structural role directly ({target user, target item, other
user, other item}) instead of computing DRNL's general formula. This is not
a rederivation of DRNL and would need revisiting if `MAX_HOP` below is ever
increased past 1.

In [ ]:
import os

FAST_SMOKE_TEST = os.environ.get("IGMC_SMOKE_TEST") == "1"

## Setup: threads, paths, imports

In [ ]:
import sys
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

N_THREADS = os.cpu_count() or 1
torch.set_num_threads(N_THREADS)
torch.set_num_interop_threads(max(1, N_THREADS // 2))
print(f"CPU cores detected: {N_THREADS}")

ROOT = Path.cwd().resolve().parent  # theGreatTry/week2 -> theGreatTry
sys.path.append(str(ROOT / "dataset loaders and cleaners"))
sys.path.append(str(ROOT / "experiments"))

from recsys_loader import load_split
from evaluate import sweep

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

## Load dataset (shared 70/20/10 protocol, seed=42)

IGMC's subgraph extraction cost scales with the graph's average degree, not
just its edge count, and this notebook extracts every subgraph with plain
Python dict lookups (see Notes at the bottom) - so we default to ML-100K.
The rest of the code is dataset-agnostic (`DATASET = "ml-1m"` etc. works
unchanged) but ML-1M/25M would take far longer to precompute.

In [ ]:
SEED = 42
DATASET = "ml-100k"
N_RATING_CLASSES = 5  # proposal Sec. 3: ratings in {1,...,5}

split = load_split(DATASET, seed=SEED)
print(f"{split.name}: {split.n_users:,} users x {split.n_items:,} items")
print(f"train={len(split.train):,}  val={len(split.val):,}  test={len(split.test):,}")

## Training-graph adjacency

Only training edges may ever be used to look up a node's neighbours - using
val/test edges here would leak the label we are trying to predict straight
into the subgraph. Ratings are stored raw (not rounded) since some datasets
allow half-star ratings; rounding only happens later, when a rating is
turned into an edge *relation type* for the R-GCN.

In [ ]:
def build_adjacency(rows):
    user_items: dict[int, dict[int, float]] = {}
    item_users: dict[int, dict[int, float]] = {}
    for u, i, r in rows:
        u, i = int(u), int(i)
        user_items.setdefault(u, {})[i] = float(r)
        item_users.setdefault(i, {})[u] = float(r)
    return user_items, item_users

## 1-hop enclosing subgraph extraction

For target pair `(u, i)`:

- nodes = `{u, i} ∪ N(u) ∪ N(i)`, where `N(u)` = other items `u` rated
  (never including `i`) and `N(i)` = other users who rated `i` (never
  including `u`) - this is what keeps the target edge itself out of the
  subgraph, so the model cannot just read off the label it's predicting;
- edges = every *training* rating between two included nodes: the "star"
  edges from `u` to `N(u)` and from `i` to `N(i)`, plus any edge that happens
  to exist between a node in `N(i)` and a node in `N(u)`. That last group is
  what actually connects the two targets' neighbourhoods - without it, `u`
  and `i` would sit in disconnected components and the GNN would degenerate
  into two independent encoders glued together only at the final MLP;
- each node is one-hot labelled purely by its structural role - never by
  identity - so the same weights generalise to users/items unseen during
  training (this is what makes the model inductive).

Popular users/items are capped to `MAX_NEIGHBORS` neighbours (uniformly
subsampled) on each side, both to bound subgraph size and because the
original IGMC implementation subsamples dense neighbourhoods the same way.

In [ ]:
MAX_NEIGHBORS = 20


def _relation(rating: float) -> int:
    """Bucket a (possibly half-star) rating into one of N_RATING_CLASSES
    edge-relation types. Only used for neighbour edges, never for the
    regression target itself."""
    return int(min(max(round(rating), 1), N_RATING_CLASSES)) - 1


def _sample(neighbors: list[int], rng: random.Random) -> list[int]:
    if len(neighbors) <= MAX_NEIGHBORS:
        return neighbors
    return rng.sample(neighbors, MAX_NEIGHBORS)


def extract_subgraph(u: int, i: int, rng: random.Random):
    """Returns (node_role, edges) for the 1-hop enclosing subgraph of (u, i).

    node_role[k] in {0,1,2,3} = {target_user, target_item, other_user, other_item}.
    Local index 0 is always u, local index 1 is always i.
    edges is a list of (local_src, local_dst, relation), one row per
    direction (the graph is undirected).
    """
    other_items = _sample([j for j in user_items.get(u, {}) if j != i], rng)
    other_users = _sample([a for a in item_users.get(i, {}) if a != u], rng)

    other_user_base = 2
    other_item_base = other_user_base + len(other_users)
    node_role = [0, 1] + [2] * len(other_users) + [3] * len(other_items)

    edges: list[tuple[int, int, int]] = []

    def add(a_local, b_local, rating):
        rel = _relation(rating)
        edges.append((a_local, b_local, rel))
        edges.append((b_local, a_local, rel))

    for k, a in enumerate(other_users):
        add(other_user_base + k, 1, item_users[i][a])  # a rated i
    for k, j in enumerate(other_items):
        add(0, other_item_base + k, user_items[u][j])  # u rated j
    for ku, a in enumerate(other_users):
        a_items = user_items.get(a, {})
        for kj, j in enumerate(other_items):
            r = a_items.get(j)
            if r is not None:
                add(other_user_base + ku, other_item_base + kj, r)  # a also rated j

    return node_role, edges

## Precompute subgraphs for model selection (train-only adjacency)

The training graph doesn't change across epochs, so extracting every
example's subgraph once up front - instead of on every forward pass - is the
difference between a runnable notebook and one that repeats the same Python
dict lookups thousands of times. This is the slow part on CPU; set
`IGMC_SMOKE_TEST=1` to sanity-check the rest of the pipeline on a small
slice first.

In [ ]:
def precompute_subgraphs(rows, rng, desc):
    out = []
    for u, i, r in tqdm(rows, desc=desc, leave=False):
        node_role, edges = extract_subgraph(int(u), int(i), rng)
        out.append((node_role, edges, float(r)))
    return out


if FAST_SMOKE_TEST:
    train_rows = split.train[:800]
    val_rows = split.val[:300]
    test_rows = split.test[:300]
else:
    train_rows, val_rows, test_rows = split.train, split.val, split.test

user_items, item_users = build_adjacency(train_rows)  # phase 1: model-selection adjacency (train only)

rng = random.Random(SEED)
sel_train_graphs = precompute_subgraphs(train_rows, rng, "train subgraphs")
sel_val_graphs = precompute_subgraphs(val_rows, rng, "val subgraphs")
print(f"train={len(sel_train_graphs):,} val={len(sel_val_graphs):,} subgraphs cached (model selection)")

## Batching subgraphs into one block-diagonal graph

Rather than pad every subgraph to the same size, we concatenate a
minibatch's node/edge lists with an index offset per graph (the same trick
PyTorch Geometric's `Batch` uses), so a whole minibatch is one relational GCN
forward pass instead of a Python loop over individually-sized graphs.

In [ ]:
def collate(graphs_batch):
    """graphs_batch: list of (node_role, edges, rating). Returns tensors for
    one combined block-diagonal graph, plus each example's target-node
    indices and its label."""
    node_roles = []
    edges_src, edges_dst, edges_rel = [], [], []
    target_u, target_i = [], []
    ratings = []
    offset = 0
    for node_role, edges, rating in graphs_batch:
        node_roles.extend(node_role)
        for a, b, rel in edges:
            edges_src.append(a + offset)
            edges_dst.append(b + offset)
            edges_rel.append(rel)
        target_u.append(offset + 0)
        target_i.append(offset + 1)
        ratings.append(rating)
        offset += len(node_role)

    return {
        "node_role": torch.tensor(node_roles, dtype=torch.long, device=device),
        "edge_src": torch.tensor(edges_src, dtype=torch.long, device=device),
        "edge_dst": torch.tensor(edges_dst, dtype=torch.long, device=device),
        "edge_rel": torch.tensor(edges_rel, dtype=torch.long, device=device),
        "target_u": torch.tensor(target_u, dtype=torch.long, device=device),
        "target_i": torch.tensor(target_i, dtype=torch.long, device=device),
        "rating": torch.tensor(ratings, dtype=torch.float32, device=device),
        "n_nodes": offset,
    }

## Relational GCN layer

Standard R-GCN update (Schlichtkrull et al. 2018): each relation `r` (rating
value) gets its own weight matrix, messages along relation-`r` edges are
averaged by that relation's in-degree, plus a separate self-loop weight so a
node always keeps some of its own signal. Looping over the (at most 5)
relations and doing one batched matmul per relation is far cheaper than
gathering a full per-edge weight matrix.

In [ ]:
class RGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim, n_relations):
        super().__init__()
        self.self_loop = nn.Linear(in_dim, out_dim, bias=True)
        self.rel_weight = nn.Parameter(torch.randn(n_relations, in_dim, out_dim) * (1.0 / in_dim**0.5))
        self.n_relations = n_relations

    def forward(self, h, edge_src, edge_dst, edge_rel, n_nodes):
        out = self.self_loop(h)
        if edge_src.numel() > 0:
            for r in range(self.n_relations):
                mask = edge_rel == r
                if not mask.any():
                    continue
                src_r = edge_src[mask]
                dst_r = edge_dst[mask]
                deg_r = torch.zeros(n_nodes, device=h.device)
                deg_r.index_add_(0, dst_r, torch.ones_like(dst_r, dtype=h.dtype))
                deg_r.clamp_(min=1.0)
                msg = h[src_r] @ self.rel_weight[r]
                msg = msg / deg_r[dst_r].unsqueeze(1)
                out = out.index_add(0, dst_r, msg)
        return torch.relu(out)

## IGMC model

Node inputs are just the 4-way structural role (an embedding lookup here is
mathematically the same as one-hot-encoding + a linear layer, just without
materialising the one-hot vectors). Every R-GCN layer's output for the two
target nodes is concatenated - this multi-scale representation is what lets
the final MLP use both very local (1-hop) and slightly wider structure.

In [ ]:
class IGMC(nn.Module):
    def __init__(self, hidden_dim=32, n_layers=3, n_relations=N_RATING_CLASSES, n_node_roles=4, dropout=0.2):
        super().__init__()
        self.node_embed = nn.Embedding(n_node_roles, hidden_dim)
        self.layers = nn.ModuleList(
            [RGCNLayer(hidden_dim, hidden_dim, n_relations) for _ in range(n_layers)]
        )
        concat_dim = hidden_dim * n_layers * 2  # n_layers per target node, 2 target nodes
        self.mlp = nn.Sequential(
            nn.Linear(concat_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, batch):
        h = self.node_embed(batch["node_role"])
        layer_outs = []
        for layer in self.layers:
            h = layer(h, batch["edge_src"], batch["edge_dst"], batch["edge_rel"], batch["n_nodes"])
            layer_outs.append(h)
        h_all = torch.cat(layer_outs, dim=-1)
        u_repr = h_all[batch["target_u"]]
        i_repr = h_all[batch["target_i"]]
        graph_repr = torch.cat([u_repr, i_repr], dim=-1)
        return self.mlp(graph_repr).squeeze(-1)

## Training / prediction helpers

In [ ]:
def evaluate_rmse(model, graphs, batch_size):
    pred = predict_igmc(model, graphs, batch_size)
    actual = np.array([g[2] for g in graphs])
    return float(np.sqrt(np.mean((pred - actual) ** 2)))


def predict_igmc(model, graphs, batch_size):
    model.eval()
    preds = []
    with torch.no_grad():
        for start in range(0, len(graphs), batch_size):
            batch = collate(graphs[start : start + batch_size])
            preds.append(model(batch).cpu().numpy())
    return np.concatenate(preds)


def train_igmc(
    train_graphs, val_graphs, hidden_dim, n_layers, lr, weight_decay, epochs, batch_size, seed,
    desc=None, show_progress=True,
):
    torch.manual_seed(seed)
    model = IGMC(hidden_dim=hidden_dim, n_layers=n_layers).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    shuffle_rng = random.Random(seed)
    idx_all = list(range(len(train_graphs)))

    history = []
    bar = tqdm(range(epochs), desc=desc or f"IGMC h={hidden_dim} L={n_layers} lr={lr} wd={weight_decay}",
               leave=False, disable=not show_progress)
    for _ in bar:
        shuffle_rng.shuffle(idx_all)
        model.train()
        for start in range(0, len(idx_all), batch_size):
            batch_idx = idx_all[start : start + batch_size]
            batch = collate([train_graphs[k] for k in batch_idx])
            opt.zero_grad()
            pred = model(batch)
            loss = torch.mean((pred - batch["rating"]) ** 2)
            loss.backward()
            opt.step()

        v_rmse = evaluate_rmse(model, val_graphs, batch_size)
        history.append(v_rmse)
        if show_progress:
            bar.set_postfix(val_rmse=f"{v_rmse:.4f}")

    return model, history

## Hyperparameter grid

Every `(config, epoch, minibatch)` step here is a real subgraph collate plus
an R-GCN forward+backward pass, which is far more expensive than Plain MF's
plain embedding lookup in `mf_vs_ours.ipynb` - so this grid is intentionally
much smaller than that notebook's, not because IGMC needs less tuning but
because each point costs orders of magnitude more. Widen it if you have the
time/GPU budget.

`BATCH_SIZE` is set well above what the model itself needs (these are small
graphs on a small GPU) because the actual bottleneck is per-step Python/CUDA
launch overhead in `collate`, not GPU compute - a bigger batch means fewer
minibatches per epoch, which is where most of the wall-clock time in this
notebook actually goes. If validation RMSE looks worse than before at the
same epoch count, that's the classic bigger-batch-means-fewer-updates
trade-off - raising `LR_GRID` a bit compensates.

In [ ]:
if FAST_SMOKE_TEST:
    HIDDEN_GRID = [16]
    LAYER_GRID = [2]
    LR_GRID = [0.01]
    WD_GRID = [1e-4]
    EPOCHS = 2
    BATCH_SIZE = 64
else:
    HIDDEN_GRID = [16, 32]
    LAYER_GRID = [2, 3]
    LR_GRID = [0.003]
    WD_GRID = [0.0, 1e-4]
    EPOCHS = 15
    BATCH_SIZE = 512

igmc_configs = [
    {"hidden_dim": h, "n_layers": l, "lr": lr, "weight_decay": wd}
    for h in HIDDEN_GRID for l in LAYER_GRID for lr in LR_GRID for wd in WD_GRID
]
print(f"IGMC configs: {len(igmc_configs)} | epochs each: {EPOCHS}")

## Grid search

In [ ]:
igmc_results = []
pbar = tqdm(igmc_configs, desc="IGMC grid search")
t0 = time.time()
for cfg in pbar:
    model, history = train_igmc(
        sel_train_graphs, sel_val_graphs, epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED,
        show_progress=False, **cfg,
    )
    igmc_results.append({**cfg, "val_rmse": history[-1], "model": model, "history": history})
    pbar.set_postfix(val_rmse=f"{history[-1]:.4f}")
print(f"IGMC grid search took {time.time() - t0:.1f}s")

best_igmc = min(igmc_results, key=lambda r: r["val_rmse"])
print("Best IGMC config:", {k: v for k, v in best_igmc.items() if k not in ("model", "history")})

## Refit on train+val, evaluate once on test

Model selection above only ever used a train-only adjacency, so validation
subgraphs never saw their own (or each other's) edges. For the final fit we
rebuild the adjacency from train+val combined and re-extract every subgraph
against it - exactly like `mf_vs_ours.ipynb`'s `trainval_u/i/r` step - so
test subgraphs can benefit from val's ratings appearing as neighbourhood
edges, the same way they would once the model is deployed and val is just
more history. Test's own edges are still never in the adjacency.

In [ ]:
trainval_rows = np.concatenate([train_rows, val_rows], axis=0)
user_items, item_users = build_adjacency(trainval_rows)  # phase 2: refit adjacency (train+val)

rng2 = random.Random(SEED)
trainval_graphs = precompute_subgraphs(trainval_rows, rng2, "trainval subgraphs")
final_test_graphs = precompute_subgraphs(test_rows, rng2, "test subgraphs")

best_cfg = {k: v for k, v in best_igmc.items() if k not in ("val_rmse", "model", "history")}
final_igmc, _ = train_igmc(
    trainval_graphs, final_test_graphs, epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED,
    desc="Refit IGMC", **best_cfg,
)

## Selective RMSE (Table 1's `IGMC (rand)` / `IGMC (supp)` rows)

IGMC has no built-in uncertainty signal (same situation as Plain MF), so
both naive abstention rules from the proposal are scored against its
predictions: random abstention, and abstention on the test user's train+val
support count (fewer ratings = treated as less reliable).

In [ ]:
actual = np.array([g[2] for g in final_test_graphs])
test_user_idx = test_rows[:, 0].astype(int)
trainval_user_idx = trainval_rows[:, 0].astype(int)

igmc_pred = predict_igmc(final_igmc, final_test_graphs, BATCH_SIZE)

rng_score = np.random.default_rng(SEED)
random_score = rng_score.random(len(actual))

n_users_total = int(max(trainval_user_idx.max(), test_user_idx.max())) + 1
support = np.bincount(trainval_user_idx, minlength=n_users_total)
support_score = -support[test_user_idx].astype(np.float64)  # more support = more reliable

table = {
    "IGMC (rand)": sweep(igmc_pred, actual, random_score),
    "IGMC (supp)": sweep(igmc_pred, actual, support_score),
}

rows_out = []
for name, r in table.items():
    row = {"method": name}
    row.update({f"p={p}": round(v, 4) for p, v in r.items()})
    rows_out.append(row)

results_df = pd.DataFrame(rows_out).set_index("method")
results_df

## Plot: selective RMSE vs. abstention rate

In [ ]:
plt.figure(figsize=(7, 5))
for name, r in table.items():
    xs = sorted(r.keys())
    ys = [r[x] for x in xs]
    plt.plot(xs, ys, marker="o", label=name)
plt.xlabel("Abstention rate p")
plt.ylabel("Selective RMSE")
plt.title(f"IGMC selective RMSE vs abstention rate ({DATASET})")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Notes / next steps

- **Runtime**: pure-Python subgraph extraction is the bottleneck, not the
  R-GCN itself - on ML-100K this notebook should run end-to-end in a few
  minutes. ML-1M/25M would need a real batched graph library (e.g. PyTorch
  Geometric, with GPU-resident neighbour sampling) to be tractable, since
  every subgraph here is built with plain Python dict lookups on CPU.
- `MAX_NEIGHBORS` caps how many neighbours each side samples; this trades
  subgraph fidelity for tractability the same way the original IGMC
  implementation subsamples dense neighbourhoods.
- The node-role labelling here is exact for `h=1` (see intro markdown) but
  does not generalise to deeper hops - a genuine DRNL implementation would
  be needed to try `h=2`.
- `Ours (U_hat)` / `Plain MF` rows live in `experiments/mf_vs_ours.ipynb`;
  `SoftImpute` / `AutoRec` / `UAIMC` rows are still open, per proposal
  Table 1.